# 第 2 章 · Agent 详解与模型接入

> 第 1 章我们用三个参数（`name / model / instruction`）创建了 Agent。本章把 `Agent` 的**完整参数面**讲透，并深入 ADK 的**模型接入机制**——理解它，你就能在 DeepSeek、Gemini、OpenAI、本地模型之间自由切换。

---

## 1. Agent（LlmAgent）参数全景

ADK 中 `Agent` 是 `LlmAgent` 的别名，核心参数如下：

| 参数 | 类型 | 作用 | 教学类比 |
|---|---|---|---|
| `name` | str | 智能体唯一标识 | 员工工号 |
| `model` | str / BaseLlm | 底层 LLM | 员工的大脑型号 |
| `instruction` | str | 系统指令（人设、规则、输出格式） | 岗位说明书 |
| `description` | str | 一句话能力描述（**供上级 Agent 做转移决策**） | 简历上的一句话简介 |
| `tools` | list | 可用工具（第 3 章） | 装备栏 |
| `sub_agents` | list | 下级智能体（第 4 章） | 直属下属 |
| `output_key` | str | 把最终回复**写入 Session 状态**的键名 | 工作成果的归档位置 |
| `output_schema` | Pydantic BaseModel | 强制结构化输出 | 规定好的汇报模板 |
| `generate_content_config` | GenerateContentConfig | 温度等生成参数 | 性格旋钮（严谨 vs 发散） |
| `planner` / `code_executor` | — | 规划器 / 代码执行器 | 进阶装备（第 6 章提及） |
| `before/after_*_callback` | callable | 生命周期钩子（第 6 章） | 审批与复盘流程 |

```mermaid
classDiagram
    class Agent {
        +str name
        +BaseLlm model
        +str instruction
        +str description
        +list tools
        +list sub_agents
        +str output_key
        +BaseModel output_schema
        +run_async() Event流
    }
    class BaseLlm {
        <<interface>>
    }
    class LiteLlm {
        +model: "deepseek/deepseek-chat"
    }
    class Gemini {
        +model: "gemini-2.x"
    }
    Agent --> BaseLlm : 依赖抽象
    LiteLlm ..|> BaseLlm
    Gemini ..|> BaseLlm
```

> 📌 **架构洞察**：`Agent` 依赖的是 `BaseLlm` **抽象接口**，而不是某个具体模型。这就是 ADK"Google 血统却模型中立"的原因——LiteLLM 只是 `BaseLlm` 的一种实现。

---

## 2. 模型接入机制：LiteLLM 适配层

```mermaid
flowchart LR
    A["ADK Agent"] -->|BaseLlm 接口| L["LiteLlm 适配器"]
    L -->|"deepseek/deepseek-chat"| DS["🐋 DeepSeek"]
    L -->|"openai/gpt-4o"| OA["OpenAI"]
    L -->|"anthropic/claude-*"| CL["Claude"]
    L -->|"ollama/llama3"| OL["本地 Ollama"]
    A -->|原生| G["Gemini / Vertex AI"]
    style L fill:#fef7e0,stroke:#fbbc04,stroke-width:2px
```

三个要点：

1. **模型名规则**：`provider/model`。LiteLLM 按 provider 前缀路由到对应服务商；`deepseek/` 前缀会自动读取 `DEEPSEEK_API_KEY`；
2. **切换模型 = 换一行字符串**，Agent 的其他代码完全不动；
3. **也能直接传字符串**：`Agent(model="gemini-2.0-flash")` 走 Google 原生路径；传 `LiteLlm(...)` 对象走适配层。

先用第 1 章的方式快速重建运行环境：


In [1]:
import os
assert os.environ.get("DEEPSEEK_API_KEY"), "请先设置 DEEPSEEK_API_KEY"

from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

APP, USER = "adk_ch02", "student"

async def run_once(agent, query, session_id="demo", verbose=False):
    """通用一次性问答：新建 Runner+Session，返回最终文本（可选打印事件流）。"""
    ss = InMemorySessionService()
    await ss.create_session(app_name=APP, user_id=USER, session_id=session_id)
    runner = Runner(agent=agent, app_name=APP, session_service=ss)
    msg = types.Content(role="user", parts=[types.Part(text=query)])
    final = ""
    async for ev in runner.run_async(user_id=USER, session_id=session_id, new_message=msg):
        if verbose and ev.content and ev.content.parts:
            for p in ev.content.parts:
                if p.text: print(f"  [event:{ev.author}] {p.text[:80]}")
        if ev.is_final_response() and ev.content and ev.content.parts:
            final = ev.content.parts[0].text
    # 顺便返回 session 便于查看状态
    session = await ss.get_session(app_name=APP, user_id=USER, session_id=session_id)
    return final, session

print("✅ 运行环境就绪")


✅ 运行环境就绪


---

## 3. `instruction`：不止是一段静态文字

### 3.1 基础用法——人设 + 规则 + 输出格式

好的 instruction 通常包含三块：**角色**、**行为准则**、**输出要求**。支持多行字符串：


In [2]:
poet = Agent(
    name="poet",
    model=LiteLlm(model="deepseek/deepseek-chat"),
    instruction="""你是一位精通中国古典诗词的文学老师。
行为准则：
1. 赏析诗词时先给白话翻译，再给艺术点评；
2. 点评不超过 100 字；
3. 结尾推荐一首风格相近的诗。
""",
    description="中国古典诗词赏析老师",
)

text, _ = await run_once(poet, "赏析一下「大漠孤烟直，长河落日圆」。")
print(text)


23:01:58 - LiteLLM:WARNING: get_model_cost_map.py:289 - LiteLLM: Failed to fetch remote model cost map from https://raw.githubusercontent.com/BerriAI/litellm/main/model_prices_and_context_window.json: The read operation timed out. Falling back to local backup.


这两句出自王维《使至塞上》。白话翻译是：广阔沙漠中，一缕孤烟笔直升起；黄河尽头，落日浑圆悬挂天边。

艺术点评：以“直”状烟，凸显荒漠无风之静；以“圆”绘日，尽显苍茫辽远之壮。白描如画，意境雄浑，边塞之景跃然纸上。

风格相近的诗，推荐王维《终南别业》中“行到水穷处，坐看云起时”，同具旷达禅意。


### 3.2 状态插值——让 instruction "活"起来

`instruction` 中可以用 **`{key}`** 引用 **Session 状态**里的值，运行时才渲染。这是实现"千人千面"人设的关键技巧：


In [3]:
greeter = Agent(
    name="greeter",
    model=LiteLlm(model="deepseek/deepseek-chat"),
    instruction="你正在和 {username} 对话，TA 的母语是 {native_lang}。请用对方的母语打招呼，并自我介绍。",
    description="多语言问候助手",
)

# 预置 session 状态：initial_state 会在创建会话时写入
ss = InMemorySessionService()
await ss.create_session(app_name=APP, user_id=USER, session_id="i18n",
                        state={"username": "小明", "native_lang": "中文"})
runner = Runner(agent=greeter, app_name=APP, session_service=ss)
msg = types.Content(role="user", parts=[types.Part(text="开始吧")])
async for ev in runner.run_async(user_id=USER, session_id="i18n", new_message=msg):
    if ev.is_final_response():
        print(ev.content.parts[0].text)


你好！我是多语言问候助手，很高兴认识你！请问有什么可以帮你的吗？😊


> ⚠️ 注意：`{key}` 是**指令注入**，渲染发生在发送给 LLM 之前；改状态后再触发新一轮，指令内容会随之变化。这与 LangChain 的 `PromptTemplate` 思想一致，但 ADK 把它与 Session 状态直接打通了。

---

## 4. `output_key`：把回复存进 Session 状态

默认情况下，Agent 的回复只是事件流里的一串 Event。加上 `output_key`，最终文本会**自动写入 session.state**，供后续 Agent 或你的代码消费——这是多智能体"接力赛"里传递接力棒的标准动作（第 4 章会大量使用）。


In [4]:
summarizer = Agent(
    name="summarizer",
    model=LiteLlm(model="deepseek/deepseek-chat"),
    instruction="把用户给的文本总结成一句话。",
    description="一句话总结器",
    output_key="summary",   # ← 关键：最终回复存入 state["summary"]
)

text, session = await run_once(summarizer, "人工智能正在深刻改变软件开发的方式，从代码补全到自动测试，再到今天的智能体协同编程，开发者的角色正从『写代码的人』转变为『指挥智能体的人』。")
print("Agent 回复：", text)
print("─" * 50)
print("Session 状态中已归档：", session.state.get("summary"))


Agent 回复： 人工智能正将开发者从代码编写者转变为指挥智能体的人。
──────────────────────────────────────────────────
Session 状态中已归档： 人工智能正将开发者从代码编写者转变为指挥智能体的人。


---

## 5. `generate_content_config`：给模型的"性格旋钮"

温度（temperature）控制随机性：`0` 严谨确定、`1+` 发散创意。ADK 统一用 `GenerateContentConfig` 表达生成参数：


In [5]:
from google.genai.types import GenerateContentConfig

creative = Agent(
    name="creative",
    model=LiteLlm(model="deepseek/deepseek-chat"),
    instruction="给产品起 3 个中文名，每个附一句 slogan。",
    description="品牌创意命名师",
    generate_content_config=GenerateContentConfig(temperature=1.2, max_output_tokens=500),
)

text, _ = await run_once(creative, "产品：一款帮助程序员管理久坐健康的 App。")
print(text)


好的，针对“帮助程序员管理久坐健康的 App”，我从不同角度为你构思了 3 个中文名及对应口号。

---

**方案一：码上动**
*   **核心理念**：谐音“马上动”，直接呼应程序员“码”字的日常，强调使用 App 后立刻起身活动的行动指令。
*   **Slogan**：**码上动，活力不宕机。**

---

**方案二：坐·立·行**
*   **核心理念**：以三个身体状态的动作为名，简洁清晰，暗示 App 的核心功能就是引导你在久坐中切换状态。名字有韵律感，像一句行为准则。
*   **Slogan**：**坐·立·行，代码人生更从容。**

---

**方案三：键程**
*   **核心理念**：将“键盘”与“里程”结合，隐喻每一次敲击代码的时光都应有健康的距离管理。比前两个更文艺，适合偏健康生活方式类的品牌调性。
*   **Slogan**：**键程千万里，健康零距离。**


---

## 6. `output_schema`：强制结构化输出

业务系统里，我们经常要的不是"一段话"，而是**一个可以入库的 JSON**。`output_schema` 接收一个 Pydantic 模型，ADK 会约束 LLM 输出对应结构。

> ⚠️ **两条使用限制**：
> 1. `output_schema` 生效时，该 Agent **不能同时使用 tools 或向其他 Agent 转移**（框架要把"输出通道"完全锁定给结构化结果）；
> 2. 它依赖底层 API 支持 **JSON Schema 模式**（`response_format: json_schema`）。Gemini 原生支持；**DeepSeek 目前仅支持 `json_object` 模式**，直接使用 `output_schema` 会报 `This response_format type is unavailable now`——这是模型方的限制，不是 ADK 的 bug。

在 Gemini 上的标准写法（示意，本环境不执行）：

```python
reviewer = Agent(
    name="reviewer",
    model="gemini-2.0-flash",          # Gemini 原生支持 json_schema
    instruction="你是书评编辑，根据用户描述生成书评。",
    output_schema=BookReview,           # ← 强制输出 BookReview 结构
    output_key="review_json",
)
```

**在 DeepSeek 下的等价工程做法**：用 instruction 明确 JSON 契约 + 代码端解析校验。这反而揭示了一个朴素真理——**结构化输出 = 提示词契约 + 解析校验**，模型原生支持只是让成功率接近 100%：


In [6]:
from pydantic import BaseModel, Field
import json, re

class BookReview(BaseModel):
    title: str = Field(description="书名")
    rating: int = Field(description="评分，1-5 的整数")
    keywords: list[str] = Field(description="3 个关键词")
    one_line: str = Field(description="一句话点评")

reviewer_ds = Agent(
    name="reviewer_ds",
    model=LiteLlm(model="deepseek/deepseek-chat"),
    instruction="""你是书评编辑。严格输出如下 JSON（不要输出任何其他文字、不要用 markdown 代码块）：
{"title": "书名", "rating": 1到5的整数, "keywords": ["关键词1", "关键词2", "关键词3"], "one_line": "一句话点评"}""",
    description="书评结构化生成器（DeepSeek 兼容版）",
    output_key="review_json",
)

text, session = await run_once(reviewer_ds, "《三体》：地球文明与三体文明的史诗级碰撞，硬核的物理设定与冷酷的宇宙社会学让人手不释卷。")
print("原始输出：", text[:120], "...")

# 防御性解析：剥离可能的 markdown 围栏后用 Pydantic 校验
cleaned = re.sub(r"^```(json)?|```$", "", text.strip(), flags=re.M).strip()
review = BookReview(**json.loads(cleaned))
print("─" * 50)
print(f"✅ 解析成功：{review.title} | 评分 {review.rating}/5 | 关键词 {review.keywords}")


原始输出： {"title": "三体", "rating": 5, "keywords": ["科幻", "宇宙社会学", "文明冲突"], "one_line": "硬核科幻与冷酷哲思的完美碰撞，宇宙尺度的文明史诗。"} ...
──────────────────────────────────────────────────
✅ 解析成功：三体 | 评分 5/5 | 关键词 ['科幻', '宇宙社会学', '文明冲突']


> 💡 **与 LangChain 对照**：LangChain 的 `llm.with_structured_output(Model)` 可通过 `method="function_calling"` 改走**工具调用通道**（DeepSeek 支持），因此它在 DeepSeek 上可用（LangChain 线路第 2 章有实测）——同样是结构化输出，两个框架的底层通道不同，模型兼容性也因此不同，选型时值得实测。

---

## 7. `description`：为多智能体埋下伏笔

`description` 在当前 Agent 自己运行时几乎不起作用，它的舞台在**第 4 章**：当一个父 Agent 需要决定"把任务转交给哪个子 Agent"时，它读的就是每个子 Agent 的 `description`。因此：

> ✍️ **写作准则**：description 要像"给调度系统看的岗位简介"——写清**擅长什么、何时该被选中**，而不是泛泛的"一个助手"。

```python
# ❌ 差的 description
description="一个很有用的助手"

# ✅ 好的 description
description="处理所有与订单退款相关的请求，包括查询退款进度、发起退款申请"
```

---

## 8. 与 LangChain 对照 🔄

| ADK 概念 | LangChain / LangGraph 对应 | 差异点评 |
|---|---|---|
| `Agent(instruction=...)` | `create_agent(model, system_prompt=...)` | 几乎一一对应 |
| `LiteLlm(model="deepseek/...")` | `ChatOpenAI(base_url=...)` 等数十种 ChatModel | LangChain 集成数量更多；ADK 借 LiteLLM 追平 |
| `output_key` 写入 state | LangGraph 节点函数返回 `{"key": value}` 更新 State | ADK 声明式一行；LangGraph 显式但灵活 |
| `output_schema=Model` | `llm.with_structured_output(Model)` | 等价 |
| `generate_content_config` | `ChatOpenAI(temperature=...)` 构造参数 | ADK 把生成参数与 Agent 绑定，LangChain 与模型绑定 |
| instruction `{state}` 插值 | `ChatPromptTemplate` 变量 | ADK 直接读 Session state，少一层模板组装 |

---

## 📌 本章要点回顾

- `Agent` 依赖 `BaseLlm` 抽象 → 换模型只需换 `LiteLlm(model="provider/name")` 一行；
- `instruction` 支持 `{key}` 从 Session 状态插值，实现动态人设；
- `output_key` 把回复归档进状态——多智能体接力的基础；
- `output_schema` 强制 JSON 结构输出，代价是禁用工具与转移；
- `description` 是写给"调度者"看的，第 4 章多智能体的关键。

> ➡️ 下一章：[03-工具系统](03-工具系统.ipynb) —— 让 Agent 从"会说"进化到"会做"。
